# **Section 2: Revision -Visualisation, functions, and grouping tables - Part 2**

## **Working with a real data file**

Part 1 built every table inline so you could edit the numbers. Real data does not
arrive that way. It arrives as a file that has more columns than you want, column
names nobody would choose, and rows that are not what they appear to be.

Getting a file into a usable state is most of the work in a real analysis, and it
is the same four steps every time:

1. **Load** it and look at it
2. **Select** the columns you actually need
3. **Relabel** anything unreadable
4. **Filter out** rows that are not real observations

This notebook needs `nc-est2019-agesex-res.csv` in the same folder.
It is the US Census Bureau's population estimates by age and sex, a real
file, downloaded as published.

**Where this is used.** `select`, `drop`, `relabeled`, `where` and `sort` are
used throughout **Project 1**. Note the spelling: the method is `relabeled`,
with one `l`.


In [ ]:
# Run this cell first -- the install takes about a minute in the browser
%pip install -q datascience ipywidgets

# pyodide_http is only present in JupyterLite -- ignored elsewhere
try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

---

## **Contents**

1. [Load it, and look at it](#1)
2. [Select the columns you need](#2)
3. [Relabel for readability](#3)
4. [Filter out rows that are not observations](#4)
5. [Rescale so the numbers can be read](#5)
6. [Comparing two groups from the same file](#6)
7. [**>>Quick questions<<**](#7)
8. [**>>Self-check<<**](#8)
9. [The recipe, and what to watch for](#9)


---

<a id='1'></a>
## **1. Load it, and look at it**

`Table.read_table(filename)` reads a CSV and returns a table. The file must be in
the same folder as this notebook, or you get a `FileNotFoundError`.

**Always look at what you loaded before doing anything with it.**


In [ ]:
full = Table.read_table('nc-est2019-agesex-res.csv')
full.show(5)

In [ ]:
# How big is it, and what are the columns actually called?
print(full.num_rows, 'rows')
print(full.num_columns, 'columns')
full.labels

Population estimates by age and sex, one column per year from 2010 to 2019.
Three things are already awkward:

- **14 columns**, and you want three of them
- `POPESTIMATE2019` is how a statistical agency names things, not how you want
  to read it on an axis label
- `CENSUS2010POP` and `ESTIMATESBASE2010` are two different kinds of 2010 figure,
  and you would have to read the documentation to know which to use

Real files are like this. The columns exist because somebody else needed them.


---

<a id='2'></a>
## **2. Select the columns you need**

`.select(...)` keeps only the columns you name, in the order you name them. It
returns a **new** table, the original is untouched.

Cutting down early makes everything afterwards easier to read.


In [ ]:
partial = full.select('SEX', 'AGE', 'POPESTIMATE2019')
partial.show(5)

In [ ]:
# The original still has all 14 columns -- .select did not change it
full.labels

> `.drop(...)` is the mirror image: name the columns to **remove** rather than
> keep. Use whichever needs less typing, `.select` when you want a few of many,
> `.drop` when you want most of them.


---

<a id='3'></a>

## **3. Relabel for readability**

There are **two** methods, and the difference is one you have met before.

- `relabeled(old, new)` returns a **new** table. The original keeps its old name.
- `relabel(old, new)` changes the table **in place**. The original is renamed.

`old` can be the current label or the column's position, counting from 0.

This is cosmetic, and it matters. You will type these names dozens of times, and
they end up as axis labels on every plot you draw.

**Your reference sheet lists `relabeled` only.** Use that one. `relabel` exists in
the library and is shown below so you can see what "in place" means, but it is not
on the sheet and you will not need it.

In [ ]:
# relabeled: the ORIGINAL is untouched
us_pop = partial.relabeled(2, '2019')
print('the copy  :', us_pop.labels)
print('the original:', partial.labels, '  <- still POPESTIMATE2019')

In [ ]:
# relabel: the ORIGINAL is changed, and nothing new is returned
scratch = partial.select(0, 1, 2)
scratch.relabel(2, '2019')
print('after relabel:', scratch.labels, '  <- scratch itself changed')

This is the same distinction as `np.append`, and as `select` and `where`: most
table methods hand you a **new** table and leave the original alone. `relabel` is
one of the few that does not.

Use `relabeled` unless you have a reason to want the change in place.

The copy is called `us_pop`, and it is what the rest of this notebook uses.

---

<a id='4'></a>

## **4. Filter out rows that are not observations**

This is the step people miss, and it is the one that silently ruins results.

Look at the values in each column before assuming you know what they mean.


In [ ]:
# What values does SEX actually take?
us_pop.group('SEX')

In [ ]:
# And AGE -- look at the largest values
us_pop.sort('AGE', descending=True).show(6)

Two traps, both common in official statistics:

**`AGE` of 999 is not an age.** It is the row holding the total across all ages.

**`SEX` of 0 is not a sex.** It is the total across both.

Neither is flagged. Nothing errors. If you leave them in and add up the
population column, you count everybody roughly **four times**, once as an
individual row, once in the age total, once in the sex total, and once in the
row that is both.

A value used as a marker rather than a measurement is called a **sentinel
value**. Look for them in any file you did not create yourself.


In [ ]:
# What the error costs, in numbers
everything = sum(us_pop.column('2019'))
print('Adding up every row:      ', f'{everything:,}')

real_rows = us_pop.where('AGE', are.below(999)).where('SEX', are.above(0))
print('Adding up only real rows: ', f'{sum(real_rows.column("2019")):,}')
print()
print('Overcounted by a factor of', round(everything / sum(real_rows.column('2019')), 1))

### **Keeping the total row on purpose**

Sometimes the sentinel is exactly what you want. To get one row per age with
**both sexes combined**, keep `SEX` 0 and drop `AGE` 999.


In [ ]:
by_age = us_pop.where('AGE', are.below(999)).where('SEX', 0)
by_age.show(5)

In [ ]:
print(by_age.num_rows, 'rows -- one per age from 0 to 100')
print('Total population:', f"{sum(by_age.column('2019')):,}")

---

<a id='5'></a>

## **5. Rescale so the numbers can be read**

58 million is hard to read on an axis, and impossible to compare by eye. Divide
by a million and say so in the column name.

`1e6` is Python's shorthand for 1000000, clearer than counting zeros.


In [ ]:
by_age = by_age.with_columns('Millions', by_age.column('2019') / 1e6)
by_age.show(5)

### **Now the plot is worth drawing**

`AGE` is ordered, so this is a line plot.


In [ ]:
by_age.plot('AGE', 'Millions')
plots.title('US population by age, 2019')
plots.show()

The shape carries real information, and it is not smooth. There is a dip around
ages 55 to 60 with bulges either side, the trough between the post-war baby boom
and the generation after it. Births in the late 1960s and early 1970s fell, and
that cohort is still visible in the data fifty years later.

There is also a spike at the far right. Age 100 is the last row in the file, so
everyone aged 100 **or older** is counted there, roughly 100,000 people, against
57,000 aged 99. That is another sentinel, hiding in plain sight as a plausible
age.

Whether you can see any of this depends on having removed the 999 rows first.
Try re-running the plot without the filter.


In [ ]:
# The same plot WITHOUT filtering -- one enormous spike at 999 flattens everything
us_pop.where('SEX', 0).plot('AGE', '2019')
plots.title('The same data, sentinel rows left in')
plots.show()

---

<a id='6'></a>
## **6. Comparing two groups from the same file**

The whole point of keeping `SEX` was to be able to split by it.


In [ ]:
males   = us_pop.where('AGE', are.below(999)).where('SEX', 1)
females = us_pop.where('AGE', are.below(999)).where('SEX', 2)

sexes = Table().with_columns(
    'Age',     males.column('AGE'),
    'Male',    males.column('2019') / 1e6,
    'Female',  females.column('2019') / 1e6)
sexes.show(5)

In [ ]:
# .plot with only an x column plots every remaining column as its own line
sexes.plot('Age')
plots.title('US population by age and sex, 2019')
plots.show()

In [ ]:
# The ratio is easier to read than two nearly-identical lines
sexes = sexes.with_columns('Females per male', sexes.column('Female') / sexes.column('Male'))
sexes.plot('Age', 'Females per male')
plots.title('Females per male, by age')
plots.show()

Slightly more boys than girls are born, the ratio starts near 0.95, and the
lines cross at about **age 38**. After that women outnumber men, and by age 100
there are more than three women for every man.

None of that was visible in the two overlapping lines above. The right *derived*
quantity often shows what the raw one hides, a point that returns in section 6,
when residuals reveal what a scatter plot does not.


---

<a id='7'></a>
## **7. Quick questions**

Five of them. Set `my_answer` to a letter and run the cell.
A wrong answer gets a nudge so you can try again; a right one gets the reason.

These come before the written questions below on purpose: they are quicker,
and they check the things people most often get wrong.

In [ ]:
# Run this once. mcq.py must be in the same folder as this notebook.
from mcq import check_answer, show_answer

**M1.** A column of ages contains the value 999. What is it most likely to be?

**a)** a data entry error  
**b)** the oldest person in the file  
**c)** a sentinel, standing for something other than an age  
**d)** an age in months  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w2b_m1', my_answer)

**M2.** Which one leaves the original table unchanged?

**a)** `tbl.relabel('old', 'new')`  
**b)** `tbl.relabeled('old', 'new')`  
**c)** both of them  
**d)** neither of them  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w2b_m2', my_answer)

**M3.** You filter a table and want to be sure the filter did what you meant. What is the simplest check?

**a)** compare `num_rows` before and after  
**b)** run `.labels`  
**c)** sort the result  
**d)** run the filter twice  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w2b_m3', my_answer)

**M4.** Why does `tbl.relabelled('a', 'b')` fail?

**a)** the column does not exist  
**b)** it needs a third argument  
**c)** you must assign the result  
**d)** there is no such method; it is `relabeled`  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w2b_m4', my_answer)

**M5.** What does `.show(5)` tell you that `.num_rows` does not?

**a)** how many rows there are  
**b)** the column names and what the values look like  
**c)** whether the file loaded  
**d)** the number of columns  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w2b_m5', my_answer)

---

<a id='8'></a>
## **8. Self-check**

Answer these without scrolling back. Reveal each answer only after you have committed to one.

**Q1.** You load a file and one column of ages contains the value 999. What is that likely to be, and why is taking the average before checking a mistake?

<details>
<summary><strong>Answer</strong></summary>

It is almost certainly a <strong>sentinel</strong>: a code standing for missing or unknown, not a real age. Averaging with it in pulls the answer upwards by an arbitrary amount. Sort each numeric column and look at both extremes before computing anything.

</details>

**Q2.** What does `.show(5)` tell you that `.num_rows` does not?

<details>
<summary><strong>Answer</strong></summary>

It shows the actual column names, the types of the values, and whether the file has totals or footnotes mixed in with the data. <code>num_rows</code> only tells you how many rows there are, which does not tell you whether they are observations.

</details>

**Q3.** You filter a table and want to be sure the filter did what you meant. What is the simplest check?

<details>
<summary><strong>Answer</strong></summary>

Compare <code>num_rows</code> before and after. If you expected to drop a handful of rows and lost half the table, the condition is wrong. It costs one line and catches the most common filtering mistake.

</details>

**Q4.** When would you use `select` and when `drop`?

<details>
<summary><strong>Answer</strong></summary>

<code>select</code> names the columns you are <strong>keeping</strong>, <code>drop</code> names the ones you are <strong>removing</strong>. Use whichever gives the shorter list. <code>select</code> is safer for a wide file, because a column you did not anticipate will not survive by accident.

</details>

**Q5.** What is the difference between `tbl.relabel('old', 'new')` and `tbl.relabeled('old', 'new')`? And why is `tbl.relabelled('old', 'new')` an error?

<details>
<summary><strong>Answer</strong></summary>

<code>relabeled</code> returns a <strong>new</strong> table and leaves the original alone. <code>relabel</code> changes the original <strong>in place</strong>. Most table methods behave like the first; <code>relabel</code> is one of the few that does not.

<code>relabelled</code>, with two <code>l</code>s, is not a method at all and raises an <code>AttributeError</code>. The <code>datascience</code> library uses the American spelling, which is one of the few places in this course where it is the correct one.

</details>

**Q6.** You want rows where salary is below 30000. Why is `tbl.where('Salary', 30000)` not that?

<details>
<summary><strong>Answer</strong></summary>

Given a bare value, <code>where</code> keeps rows <strong>equal</strong> to it, so you would get only people earning exactly 30000. For a comparison you need a predicate: <code>tbl.where('Salary', are.below(30000))</code>.

</details>

---

<a id='9'></a>
## **9. The recipe, and what to watch for**

Every time you open a file you have not used before:

```python
tbl = Table.read_table('file.csv')   # 1. load
tbl.show(5)                          #    LOOK at it
tbl.labels                           #    what are the columns called?
tbl.num_rows                         #    how big?

tbl.group('some_column')             # 2. what values does each column take?

small = tbl.select('a', 'b', 'c')    # 3. keep what you need
small = small.relabeled('UGLY', 'x') # 4. rename; note the assignment back
clean = small.where('x', are.below(999))   # 5. drop the non-observations
```

### **Checklist**

| Check | Why |
|---|---|
| assign the result of `relabeled` | it returns a **new** table; only `relabel` changes the original |
| `.show(5)` before anything else | You cannot filter what you have not seen |
| `.labels` | Column names are rarely what you assumed |
| `.group(col)` on each categorical column | Reveals sentinel codes like 0 or 999 |
| `.sort(col, descending=True).show(5)` on numeric columns | Reveals 999, 9999, −1 at the extremes |
| `num_rows` before and after filtering | Confirms you removed what you meant to |
| Does the total look plausible? | 1.3 billion Americans means you left the total rows in |

### **Sentinels you will meet**

| Value | Usually means |
|---|---|
| 999, 9999 | "all", "total", or "unknown" |
| 0 in a category column | "all categories combined" |
| −1 | "missing" or "not applicable" |
| Blank or `nan` | Genuinely missing |

None of these announce themselves. The only defence is looking at the data
before you compute on it.

---

### **Methods in this notebook**

| Call | Does |
|---|---|
| `Table.read_table(file)` | Reads a CSV into a table |
| `tbl.labels` | The column names, as a tuple |
| `tbl.select(...)` | Keeps only the named columns |
| `tbl.drop(...)` | Removes the named columns |
| `tbl.relabeled(old, new)` | Renames a column, by name or position |
| `tbl.where(col, are.below(n))` | Keeps rows where the condition holds |
| `tbl.with_columns(name, values)` | Adds a column |
| `tbl.plot(x)` | Plots every other column against x |
